In [1]:
import pandas as pd
import json
from sklearn.metrics import classification_report, mean_absolute_error


In [5]:
import pandas as pd
import json
from sklearn.metrics import classification_report, mean_absolute_error

def evaluate_macro_performance(gold_jsonl: str, qwen_jsonl: str):
    # Load Gold Dataset
    gold_records = []
    with open(gold_jsonl, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                gold_records.append({
                    "article_id": obj["article_id"],
                    "gold_rating": obj["rating"],
                    "gold_comment": obj.get("comment"),
                })
    df_gold = pd.DataFrame(gold_records)

    # Load Qwen Dataset
    qwen_records = []
    with open(qwen_jsonl, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                qwen_records.append({
                    "article_id": obj["article_id"],
                    "llm_rating": obj.get("rating"), # Use .get() to avoid KeyError if missing
                    "llm_comment": obj.get("comment"),
                    "status": obj.get("status", "success")
                })
    df_qwen = pd.DataFrame(qwen_records)
    df_qwen = df_qwen[df_qwen['status'] == 'success']

    # Merge datasets on article_id
    df = pd.merge(df_gold, df_qwen, on="article_id")

    # --- CRITICAL FIX: Drop any rows containing NaN in the ratings columns ---
    initial_count = len(df)
    df = df.dropna(subset=['gold_rating', 'llm_rating'])
    final_count = len(df)
    
    dropped_count = initial_count - final_count
    if dropped_count > 0:
        print(f"[⚠️ WARNING] Dropped {dropped_count} rows due to missing/NaN rating values from the LLM outputs.\n")

    # Force cast to integers to ensure perfect metric compatibility
    df['gold_rating'] = df['gold_rating'].astype(int)
    df['llm_rating'] = df['llm_rating'].astype(int)

    # 1. Compute Mean Absolute Error
    mae = mean_absolute_error(df['gold_rating'], df['llm_rating'])
    print(f"=== QUANTITATIVE METRICS ===\nMean Absolute Error (MAE): {mae:.4f}\n")

    # 2. Compute Classification Report (Macro F1)
    target_names = ['Extremely Negative (-2)', 'Negative (-1)', 'Neutral (0)', 'Positive (1)', 'Highly Positive (2)']
    present_classes = sorted(list(set(df['gold_rating']).union(set(df['llm_rating']))))
    labels = [c for c in [-2, -1, 0, 1, 2] if c in present_classes]
    names = [target_names[c+2] for c in labels]
    
    print(classification_report(df['gold_rating'], df['llm_rating'], labels=labels, target_names=names))
    
    # Save merged data for cluster and anomaly profiling
    df.to_csv('macro_evaluation_gemini.csv', index=False)
    print("Merged evaluation data saved to 'macro_evaluation_gemini.csv'")

    # # 3. Grouping the output of your merged evaluation
    # # Save merged data for cluster and anomaly profiling
    # df.to_csv('macro_evaluation_merged.csv', index=False)
    # print("Merged evaluation data saved to 'macro_evaluation_merged.csv'")

    # # --- FIX: Use .size() to get the count of items per cluster ---
    # cluster_analysis = df.groupby('cluster').size() 
    # print("\nCluster distribution:\n", cluster_analysis)

    # # Compute metrics per cluster securely
    # cluster_metrics = df.groupby('cluster').agg(
    #     mean_gold=('gold_rating', 'mean'),
    #     mean_llm=('llm_rating', 'mean'),
    #     variance_gold=('gold_rating', 'var'),
    #     mae_error=('article_id', lambda x: mean_absolute_error(df.loc[x.index, 'gold_rating'], df.loc[x.index, 'llm_rating']))
    # )
    
    # print("\n=== CLUSTER ANALYSIS ===")
    # print(cluster_metrics)


evaluate_macro_performance('gold_standard_india.jsonl', 'gemini3.1_pro_extended_india.jsonl')

=== QUANTITATIVE METRICS ===
Mean Absolute Error (MAE): 0.0400

                         precision    recall  f1-score   support

Extremely Negative (-2)       1.00      0.90      0.95        10
          Negative (-1)       0.88      0.88      0.88         8
            Neutral (0)       0.97      1.00      0.98        31
           Positive (1)       1.00      1.00      1.00         1

               accuracy                           0.96        50
              macro avg       0.96      0.94      0.95        50
           weighted avg       0.96      0.96      0.96        50

Merged evaluation data saved to 'macro_evaluation_gemini.csv'
